In [2]:
import nltk
import pandas as pd
from pprint import pprint
from collections import Counter
import random
import difflib
from wordfreq import zipf_frequency
from nltk.corpus import wordnet as wn

In [3]:
#prendi il primo synset nominale di WordNet associato alla parola "vehicle".
vehicle = wn.synset("vehicle.n.01")

print(vehicle)
print(vehicle.definition())
print()

Synset('vehicle.n.01')
a conveyance that transports people or objects



In [4]:
#Analisi esplorativa di iponomi di Vehicle nodo 1
for synset in vehicle.hyponyms():
    print(synset.name(), "-", synset.definition())

wheeled_vehicle.n.01 - a vehicle that moves on wheels and usually has a container for transporting things or people
rocket.n.01 - any vehicle self-propelled by a rocket engine
craft.n.02 - a vehicle designed for navigation in or on water or air or through outer space
bumper_car.n.01 - a small low-powered electrically powered vehicle driven on a special platform where there are many others to be dodged
steamroller.n.02 - vehicle equipped with heavy wide smooth rollers for compacting roads and pavements
sled.n.01 - a vehicle mounted on runners and pulled by horses or dogs; for transportation over snow
skibob.n.01 - a vehicle resembling a bicycle but having skis instead of wheels; the rider wears short skis for balancing
military_vehicle.n.01 - vehicle used by the armed forces


In [5]:
#Analisi esplorativa di iponomi di Vehicle nodo 2
for synset in vehicle.hyponyms():
    print("\n", synset.name(), "-", synset.definition())
    
    for child in synset.hyponyms():
        print("   ->", child.name(), "-", child.definition())


 wheeled_vehicle.n.01 - a vehicle that moves on wheels and usually has a container for transporting things or people
   -> wagon.n.04 - a child's four-wheeled toy cart sometimes used for coasting
   -> rolling_stock.n.01 - collection of wheeled vehicles owned by a railroad or motor carrier
   -> unicycle.n.01 - a vehicle with a single wheel that is driven by pedals
   -> scooter.n.02 - child's two-wheeled vehicle operated by foot
   -> bicycle.n.01 - a wheeled vehicle that has two wheels and is moved by foot pedals
   -> wagon.n.01 - any of various kinds of wheeled vehicles drawn by an animal or a tractor
   -> motor_scooter.n.01 - a wheeled vehicle with small wheels and a low-powered gasoline engine geared to the rear wheel
   -> skateboard.n.01 - a board with wheels that is ridden in a standing or crouching position and propelled by foot
   -> baby_buggy.n.01 - a small vehicle with four wheels in which a baby or child is pushed around
   -> boneshaker.n.01 - any wheeled vehicle that

In [6]:
#Analisi esplorativa di iponomi di Vehicle nodo 3
for synset in vehicle.hyponyms():
    
    print("\n", synset.name(), "-", synset.definition())

    for child in synset.hyponyms():
        
        print("   ->", child.name(), "-", child.definition())

        for grandchild in child.hyponyms():
            print("      ->", grandchild.name(), "-", grandchild.definition())


 wheeled_vehicle.n.01 - a vehicle that moves on wheels and usually has a container for transporting things or people
   -> wagon.n.04 - a child's four-wheeled toy cart sometimes used for coasting
   -> rolling_stock.n.01 - collection of wheeled vehicles owned by a railroad or motor carrier
   -> unicycle.n.01 - a vehicle with a single wheel that is driven by pedals
   -> scooter.n.02 - child's two-wheeled vehicle operated by foot
   -> bicycle.n.01 - a wheeled vehicle that has two wheels and is moved by foot pedals
      -> mountain_bike.n.01 - a bicycle with a sturdy frame and fat tires; originally designed for riding in mountainous country
      -> safety_bicycle.n.01 - bicycle that has two wheels of equal size; pedals are connected to the rear wheel by a multiplying gear
      -> velocipede.n.01 - any of several early bicycles with pedals on the front wheel
      -> ordinary.n.04 - an early bicycle with a very large front wheel and small back wheel
      -> push-bike.n.01 - a bicyc

In [ ]:
def get_nodes_depth(root, max_depth=3):
   #funzione per ricavare l'albero iniziale
   #crazione di un set (quindi synset duplicati vengono già scartati all'inizio)
   #si crea una lista con tutti i synset trovati nei vali livelli
   #si ripete il ciclo fino alla massima profondità che si è stabilita a 3
   #all'interno del ciclo: ogni volta si crea una lista per un nuovo livello di profondità
   #                       dopodichè si  crea un altro cilclo per cui per ogni nodo nella lista corrente si ricavano
   #                       i suoi iponomi e si mettono nella lista per il nuovo livello.
   #                       Poi si aggiorna il set con gli iponomi ricavati e la lista corrente diventa la lista creata a inizio ciclo.
   #l'output della funziona è il set con tutti gli elementi dell'albero fino al terzo livello di profondità
    nodes = {root}
    current_level = [root]

    for _ in range(max_depth):
        next_level = []

        for node in current_level:
            children = node.hyponyms()
            next_level.extend(children)

        nodes.update(next_level)
        current_level = next_level

    return list(nodes)

In [8]:

nodes = get_nodes_depth(vehicle, max_depth=3)


In [9]:
#funzione di filtro per evitare lemmi costituiti da più parole, parole con meno di 3 lettere e/o poco comuni
def valid_node(synset):
    lemma = synset.lemmas()[0].name()

    return (
        lemma.isalpha()
        and len(lemma) >= 3
        and zipf_frequency(lemma, "en") >= 2
    )

In [ ]:
#si crea una lista di synset a partire dal set ricavato in precedenza ma filtrando ogni elemento del set attraverso la funzione filtro
filtered_nodes = [
    node for node in nodes
    if valid_node(node)
]

In [ ]:
#si risolve eventuali problemi di polisemia: se sono stati selezionati due synset diversi con al posto 0 il medesimo lemma ( che quindi ha un valore polisemico)
#si mantiene solamente il primo che si incontra. 
filtered_nodes_unique = []
seen_lemmas = set()

for node in filtered_nodes:
    lemma = node.lemmas()[0].name()

    if lemma not in seen_lemmas:
        filtered_nodes_unique.append(node)
        seen_lemmas.add(lemma)

In [19]:
print("Nodi totali:", len(nodes))
print("Nodi dopo il filtro:", len(filtered_nodes))
print("Nodi unici:", len(filtered_nodes_unique))

Nodi totali: 156
Nodi dopo il filtro: 68
Nodi unici: 64


In [ ]:
#si escludono i figli di synset che sono stati esclusi dalla funzione di filtro
#poi si stampa graficamente la struttura dell'albero rimasto, stampando chiaramente i lemmi in posizione 0 dei synset
filtered_set = set(filtered_nodes_unique)

print("vehicle")

for synset in vehicle.hyponyms():

    if synset in filtered_set:
        print("├──", synset.lemmas()[0].name())

        for child in synset.hyponyms():

            if child in filtered_set:
                print("│   ├──", child.lemmas()[0].name())

                for grandchild in child.hyponyms():

                    if grandchild in filtered_set:
                        print("│   │   └──", grandchild.lemmas()[0].name())

vehicle
├── rocket
│   ├── missile
│   │   └── sidewinder
├── craft
│   ├── aircraft
│   ├── vessel
│   │   └── boat
│   │   └── galley
│   │   └── ship
│   │   └── yacht
│   ├── spacecraft
│   │   └── lander
│   │   └── starship
│   ├── hovercraft
├── steamroller
├── sled
│   ├── toboggan
│   ├── bobsled
│   ├── luge


In [ ]:
#creo manualmente la lista di lemmi dell'albero stampato in precedenza
final_words = [
    "vehicle",
    "rocket",
    "missile",
    "sidewinder",
    "craft",
    "aircraft",
    "vessel",
    "galley",
    "ship",
    "yacht",
    "boat",
    "hovercraft",
    "spacecraft",
    "starship",
    "lander",
    "steamroller",
    "sled",
    "bobsled",
    "luge",
    "toboggan"
]

In [ ]:
#creo lista di synset derivati da filtered:nodes_unique basati sulla presenza o meno del loro primo lemma nella lista final_words
#in questo modo ho definitivamente escluso gli iponimi figli di synset non filtrati
final_nodes = [
    node
    for node in filtered_nodes_unique
    if node.lemmas()[0].name() in final_words
]

print("Nodi finali:", len(final_nodes))

for node in final_nodes:
    print(node.name(), "->", node.lemmas()[0].name())

Nodi finali: 20
yacht.n.01 -> yacht
craft.n.02 -> craft
rocket.n.01 -> rocket
bobsled.n.01 -> bobsled
sled.n.01 -> sled
vehicle.n.01 -> vehicle
ship.n.01 -> ship
vessel.n.02 -> vessel
sidewinder.n.02 -> sidewinder
aircraft.n.01 -> aircraft
boat.n.01 -> boat
steamroller.n.02 -> steamroller
lander.n.02 -> lander
toboggan.n.01 -> toboggan
galley.n.01 -> galley
spacecraft.n.01 -> spacecraft
hovercraft.n.01 -> hovercraft
missile.n.01 -> missile
starship.n.01 -> starship
luge.n.01 -> luge


In [26]:
set(filtered_nodes_unique) == set(final_nodes)
len(filtered_nodes_unique), len(final_nodes)

(64, 20)

In [ ]:
#creo la lista per il dataset:
#al suo interno vi sono le possibili coppie dei 20 lemmi assieme alla loro distanza tassonomica    
words = [node.lemmas()[0].name() for node in final_nodes]

pairs = []

for i in range(len(final_nodes)):
    for j in range(i + 1, len(final_nodes)):

        synset1 = final_nodes[i]
        synset2 = final_nodes[j]

        word1 = words[i]
        word2 = words[j]

        distance = synset1.shortest_path_distance(synset2)

        pairs.append([word1, word2, distance])

print("Nodi:", len(final_nodes))
print("Coppie:", len(pairs))

Nodi: 20
Coppie: 190


In [23]:
import pandas as pd

df_pairs = pd.DataFrame(
    pairs,
    columns=["word1", "word2", "taxonomic_distance"]
)

print(df_pairs.head())
print("Numero di coppie:", len(df_pairs))

     word1       word2  taxonomic_distance
0  vehicle       craft                   1
1  vehicle  hovercraft                   2
2  vehicle      vessel                   2
3  vehicle      galley                   3
4  vehicle        ship                   3
Numero di coppie: 190


In [24]:
df_pairs["taxonomic_distance"].value_counts().sort_index()

taxonomic_distance
1    19
2    37
3    49
4    48
5    31
6     6
Name: count, dtype: int64

In [104]:
df_pairs.to_csv("taxonomic_pairs.csv", index=False)